# 🧠 Notebook 4: Deep Learning Pipeline (BONUS)

**Môn học:** Học Máy (CO3001) — HK I (2025-2026)  
**Nhóm:** Hoàng Xuân Bách · Nguyễn Việt Hùng · Trần Văn Hùng

---
### Mục tiêu
- Huấn luyện MLP với PyTorch
- Huấn luyện TabNet
- So sánh deep learning vs traditional ML

## 1. Import & Kiểm Tra Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, joblib
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from modules import config
from modules.models import DeepLearningTrainer
from modules.evaluation import ModelEvaluator

# Kiểm tra thư viện
TORCH_AVAILABLE  = False
TABNET_AVAILABLE = False

try:
    import torch
    TORCH_AVAILABLE = True
    print(f'✅ PyTorch  : {torch.__version__}')
except ImportError:
    print('❌ PyTorch chưa cài  → pip install torch')

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    TABNET_AVAILABLE = True
    print('✅ TabNet   : OK')
except ImportError:
    print('❌ TabNet chưa cài → pip install pytorch-tabnet')

## 2. Load Data

In [ ]:
X_train = np.load(config.FEATURES_DIR / 'X_train.npy').astype(np.float32)
X_test  = np.load(config.FEATURES_DIR / 'X_test.npy').astype(np.float32)
y_train = np.load(config.FEATURES_DIR / 'y_train.npy').astype(int)
y_test  = np.load(config.FEATURES_DIR / 'y_test.npy').astype(int)

# Tách validation set từ train set
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=config.VAL_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=y_train
)

print(f'Train : {X_tr.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')

## 3. MLP Model

### 3.1 Huấn Luyện MLP

In [ ]:
if TORCH_AVAILABLE:
    mlp_trainer = DeepLearningTrainer(model_type='MLP', verbose=True)
    mlp_model, mlp_history = mlp_trainer.train_mlp(X_tr, y_tr, X_val, y_val)
    print('\n✅ MLP training hoàn thành!')
else:
    print('⏭️  Bỏ qua — PyTorch chưa được cài.')

### 3.2 Training History

In [ ]:
if TORCH_AVAILABLE and mlp_trainer.history:
    history = mlp_trainer.history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history['train_loss'], color='royalblue', linewidth=2)
    ax1.set_title('Training Loss', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3)

    ax2.plot(history['train_acc'], color='coral', linewidth=2)
    ax2.set_title('Training Accuracy (%)', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.grid(True, alpha=0.3)

    plt.suptitle('MLP Training History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/figures/mlp_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

### 3.3 Đánh Giá MLP

In [ ]:
if TORCH_AVAILABLE:
    import torch
    mlp_model.eval()
    with torch.no_grad():
        X_test_t = torch.FloatTensor(X_test)
        outputs  = mlp_model(X_test_t)
        _, preds = outputs.max(1)
        y_pred_mlp = preds.numpy()

    mlp_acc = accuracy_score(y_test, y_pred_mlp)
    print(f'MLP Test Accuracy: {mlp_acc:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred_mlp))

## 4. TabNet Model

### 4.1 Huấn Luyện TabNet

In [ ]:
if TABNET_AVAILABLE:
    tabnet_trainer = DeepLearningTrainer(model_type='TabNet', verbose=True)
    tabnet_model = tabnet_trainer.train_tabnet(X_tr, y_tr, X_val, y_val)
    print('\n✅ TabNet training hoàn thành!')
else:
    print('⏭️  Bỏ qua — TabNet chưa được cài.')

### 4.2 Đánh Giá TabNet

In [ ]:
if TABNET_AVAILABLE:
    y_pred_tabnet = tabnet_model.predict(X_test)
    tabnet_acc = accuracy_score(y_test, y_pred_tabnet)

    print(f'TabNet Test Accuracy: {tabnet_acc:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred_tabnet))

### 4.3 TabNet Feature Importance

In [ ]:
if TABNET_AVAILABLE:
    with open(config.FEATURES_DIR / 'feature_names.txt') as f:
        feat_names = [l.strip() for l in f]

    importances = tabnet_model.feature_importances_
    top_n   = min(20, len(feat_names))
    indices = np.argsort(importances)[::-1][:top_n]

    plt.figure(figsize=(10, 8))
    plt.barh(range(top_n), importances[indices], color='teal', alpha=0.8)
    plt.yticks(range(top_n), [feat_names[i] for i in indices])
    plt.xlabel('Importance')
    plt.title('TabNet — Top Feature Importances', fontsize=13, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('../reports/figures/tabnet_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. So Sánh Tổng Thể: Traditional ML vs Deep Learning

In [ ]:
import pandas as pd

# Load kết quả traditional ML
trad_df = pd.read_csv(config.REPORTS_DIR / 'model_results.csv', index_col=0)

# Gộp kết quả DL
dl_rows = {}
if TORCH_AVAILABLE:  dl_rows['MLP']    = {'accuracy': mlp_acc,    'type': 'Deep Learning'}
if TABNET_AVAILABLE: dl_rows['TabNet'] = {'accuracy': tabnet_acc, 'type': 'Deep Learning'}

# Tạo bảng so sánh
compare = trad_df[['accuracy']].copy()
compare['type'] = 'Traditional ML'
for name, row in dl_rows.items():
    compare.loc[name] = [row['accuracy'], row['type']]

compare = compare.sort_values('accuracy', ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))
colors = ['steelblue' if t == 'Traditional ML' else 'coral'
          for t in compare['type']]
bars = ax.barh(compare.index, compare['accuracy'], color=colors, edgecolor='white', alpha=0.85)

for bar, val in zip(bars, compare['accuracy']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Traditional ML'),
                   Patch(facecolor='coral',     label='Deep Learning')]
ax.legend(handles=legend_elements, fontsize=11)
ax.set_xlabel('Test Accuracy', fontsize=12)
ax.set_title('Traditional ML vs Deep Learning — Accuracy Comparison',
             fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig('../reports/figures/ml_vs_dl_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Final Comparison ===')
print(compare.sort_values('accuracy', ascending=False).to_string())

## 6. Lưu Deep Learning Models

In [ ]:
if TORCH_AVAILABLE:
    import torch
    torch.save(mlp_model.state_dict(), config.MODELS_DIR / 'mlp_model.pth')
    print('✅ MLP model saved!')

if TABNET_AVAILABLE:
    tabnet_model.save_model(str(config.MODELS_DIR / 'tabnet_model'))
    print('✅ TabNet model saved!')

## 7. Kết Luận

*(Điền nhận xét của nhóm)*

- **Model tốt nhất:** ...
- **Deep learning có vượt trội không?** ...
- **Lý do:** ...
- **Đề xuất cho production:** ...